# 01 - Data Preparation for Fine-tuning

This notebook downloads, explores, cleans, and merges 3 Vietnamese datasets for fine-tuning.

**Datasets:**
1. CSConDa (ura-hcmut/Vietnamese-Customer-Support-QA) - 8,349 train + 1,500 test QA pairs (columns: question, answer, type)
2. Vietnamese-Ecommerce-Multi-turn-Chat (5CD-AI) - 1,482 multi-turn conversations
3. Vietnamese-Multi-turn-Chat-Alpaca (5CD-AI) - 12,697 conversations (will be filtered by relevance)

**Output:** Cleaned JSONL files in ChatML format for Llama 3.1 fine-tuning.

**IMPORTANT:** Run cell by cell. Read output at each step. Do NOT Run All.

In [ ]:
# Cell 1: Install dependencies
!pip install -q datasets pandas

In [ ]:
# Cell 2: Imports
import json
import random
from pathlib import Path
from collections import Counter

import pandas as pd
from datasets import load_dataset

random.seed(42)
OUTPUT_DIR = Path("data_prepared")
OUTPUT_DIR.mkdir(exist_ok=True)

SYSTEM_PROMPT = (
    "Bạn là nhân viên chăm sóc khách hàng của một shop thời trang nữ online. "
    "Trả lời thân thiện, lịch sự, xưng em gọi khách là chị/anh. "
    "Dùng emoji phù hợp. Trả lời ngắn gọn, đúng trọng tâm."
)

print("Ready.")

---
## Step 1: Download datasets

In [ ]:
# Cell 3: Download CSConDa
# NOTE: You must accept the agreement at https://huggingface.co/datasets/ura-hcmut/Vietnamese-Customer-Support-QA
try:
    ds_csconda = load_dataset("ura-hcmut/Vietnamese-Customer-Support-QA")
    print(f"CSConDa loaded: {ds_csconda}")
    print(f"Columns: {ds_csconda['train'].column_names}")
    print(f"Train: {len(ds_csconda['train'])} rows")
    if 'test' in ds_csconda:
        print(f"Test: {len(ds_csconda['test'])} rows")
except Exception as e:
    print(f"ERROR: {e}")
    print("Visit https://huggingface.co/datasets/ura-hcmut/Vietnamese-Customer-Support-QA to accept agreement")
    ds_csconda = None

In [ ]:
# Cell 4: Download Ecommerce
ds_ecommerce = load_dataset("5CD-AI/Vietnamese-Ecommerce-Multi-turn-Chat")
print(f"Ecommerce loaded: {ds_ecommerce}")
print(f"Columns: {ds_ecommerce['train'].column_names}")
print(f"Train: {len(ds_ecommerce['train'])} rows")

In [ ]:
# Cell 5: Download Alpaca
ds_alpaca = load_dataset("5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca")
print(f"Alpaca loaded: {ds_alpaca}")
print(f"Columns: {ds_alpaca['train'].column_names}")
print(f"Train: {len(ds_alpaca['train'])} rows")

---
## Step 2: Explore each dataset (READ BY EYE)

In [ ]:
# Cell 6: Explore CSConDa - 10 random samples
if ds_csconda is not None:
    train = ds_csconda['train']
    indices = random.sample(range(len(train)), min(10, len(train)))
    for i in indices:
        row = train[i]
        print(f"\n--- Sample {i} ---")
        for k, v in row.items():
            val = str(v)[:300]
            print(f"  {k}: {val}")
else:
    print("CSConDa not loaded - skip")

In [ ]:
# Cell 7: Explore Ecommerce - 10 random samples
train = ds_ecommerce['train']
indices = random.sample(range(len(train)), 10)
for i in indices:
    row = train[i]
    print(f"\n--- Sample {i} ---")
    print(f"  id: {row['id']}")
    convs = row['conversations']
    print(f"  num_turns: {len(convs)}")
    for turn in convs[:4]:  # show first 4 turns
        val = turn['value'][:150]
        print(f"    [{turn['from']}]: {val}")

In [ ]:
# Cell 8: Explore Alpaca - 10 random samples
train = ds_alpaca['train']
indices = random.sample(range(len(train)), 10)
for i in indices:
    row = train[i]
    print(f"\n--- Sample {i} ---")
    print(f"  id: {row['id']}")
    convs = row['conversations']
    print(f"  num_turns: {len(convs)}")
    for turn in convs[:4]:
        val = turn['value'][:150]
        print(f"    [{turn['from']}]: {val}")

---
## Step 3: Write converter for each dataset

Target format (ChatML for Llama 3.1):
```json
{"messages": [
  {"role": "system", "content": "..."},
  {"role": "user", "content": "..."},
  {"role": "assistant", "content": "..."}
], "source": "csconda"}
```

In [ ]:
# Cell 9: Converter for CSConDa
def convert_csconda(dataset) -> list[dict]:
    """Convert CSConDa QA pairs to ChatML format."""
    converted = []
    for row in dataset:
        # Inspect column names from exploration step
        # Expected: question, answer (or similar)
        question = row.get('question', row.get('Question', ''))
        answer = row.get('answer', row.get('Answer', ''))

        if not question or not answer:
            continue
        if len(question.strip()) < 3 or len(answer.strip()) < 3:
            continue

        converted.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question.strip()},
                {"role": "assistant", "content": answer.strip()},
            ],
            "source": "csconda",
        })

    return converted

if ds_csconda is not None:
    csconda_converted = convert_csconda(ds_csconda['train'])
    print(f"CSConDa converted: {len(csconda_converted)} samples")
    print(f"\nSample:")
    print(json.dumps(csconda_converted[0], ensure_ascii=False, indent=2))
else:
    csconda_converted = []
    print("CSConDa not available - skipping")

In [ ]:
# Cell 10: Converter for Ecommerce multi-turn
def convert_ecommerce(dataset) -> list[dict]:
    """Convert multi-turn conversations. Map human->user, gpt->assistant."""
    converted = []
    role_map = {"human": "user", "gpt": "assistant"}

    for row in dataset:
        convs = row['conversations']
        if not convs or len(convs) < 2:
            continue

        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        valid = True
        for turn in convs:
            role = role_map.get(turn['from'])
            if not role:
                valid = False
                break
            content = turn['value'].strip()
            if not content:
                valid = False
                break
            messages.append({"role": role, "content": content})

        if valid and len(messages) >= 3:  # system + at least 1 user + 1 assistant
            converted.append({"messages": messages, "source": "ecommerce"})

    return converted

ecommerce_converted = convert_ecommerce(ds_ecommerce['train'])
print(f"Ecommerce converted: {len(ecommerce_converted)} samples")
print(f"\nSample:")
print(json.dumps(ecommerce_converted[0], ensure_ascii=False, indent=2)[:500])

In [ ]:
# Cell 11: Converter for Alpaca (WITH FILTERING)
# Alpaca is multi-domain - we only keep samples related to shopping/products/customer service

KEEP_KEYWORDS = [
    "sản phẩm", "mua", "bán", "giá", "size", "màu", "chất liệu", "vải",
    "đổi trả", "hoàn", "ship", "giao hàng", "vận chuyển", "đơn hàng",
    "thanh toán", "khuyến mãi", "giảm giá", "voucher", "mã",
    "áo", "váy", "quần", "đầm", "chân váy", "túi", "giày", "dép",
    "thời trang", "outfit", "phối đồ", "mặc",
    "khách hàng", "shop", "cửa hàng", "tư vấn", "hỗ trợ",
    "bảo hành", "bảo quản", "giặt", "ủi",
    "review", "đánh giá", "feedback", "phàn nàn", "khiếu nại",
    "mỹ phẩm", "kem", "serum", "son", "nước hoa",
]

def is_relevant(conversations: list[dict]) -> bool:
    """Check if conversation is related to shopping/customer service."""
    text = " ".join(t['value'].lower() for t in conversations)
    return any(kw in text for kw in KEEP_KEYWORDS)

def convert_alpaca(dataset) -> list[dict]:
    """Convert Alpaca multi-turn, keeping only relevant conversations."""
    converted = []
    filtered_count = 0
    role_map = {"human": "user", "gpt": "assistant"}

    for row in dataset:
        convs = row['conversations']
        if not convs or len(convs) < 2:
            continue

        if not is_relevant(convs):
            filtered_count += 1
            continue

        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        valid = True
        for turn in convs:
            role = role_map.get(turn['from'])
            if not role:
                valid = False
                break
            content = turn['value'].strip()
            if not content:
                valid = False
                break
            messages.append({"role": role, "content": content})

        if valid and len(messages) >= 3:
            converted.append({"messages": messages, "source": "alpaca"})

    print(f"Total Alpaca rows: {len(dataset)}")
    print(f"Filtered out (not relevant): {filtered_count}")
    print(f"Kept: {len(converted)}")
    return converted

alpaca_converted = convert_alpaca(ds_alpaca['train'])
print(f"\nSample:")
if alpaca_converted:
    print(json.dumps(alpaca_converted[0], ensure_ascii=False, indent=2)[:500])

---
## Step 4: Validate converted data

In [ ]:
# Cell 12: Validate each converted dataset
def validate_dataset(data: list[dict], name: str) -> None:
    print(f"\n=== Validating {name} ===")
    print(f"Total samples: {len(data)}")

    errors = []
    for i, item in enumerate(data):
        msgs = item.get('messages', [])
        if len(msgs) < 3:
            errors.append(f"Sample {i}: too few messages ({len(msgs)})")
        if msgs[0]['role'] != 'system':
            errors.append(f"Sample {i}: first message is not system")
        for msg in msgs:
            if not msg.get('content', '').strip():
                errors.append(f"Sample {i}: empty content in {msg['role']}")
            if msg['role'] not in ('system', 'user', 'assistant'):
                errors.append(f"Sample {i}: invalid role {msg['role']}")

    if errors:
        print(f"ERRORS found: {len(errors)}")
        for e in errors[:10]:
            print(f"  {e}")
    else:
        print("All samples valid!")

    # Length stats
    user_lens = []
    asst_lens = []
    for item in data:
        for msg in item['messages']:
            if msg['role'] == 'user':
                user_lens.append(len(msg['content']))
            elif msg['role'] == 'assistant':
                asst_lens.append(len(msg['content']))

    if user_lens:
        print(f"User message length: min={min(user_lens)}, max={max(user_lens)}, avg={sum(user_lens)/len(user_lens):.0f}")
    if asst_lens:
        print(f"Assistant message length: min={min(asst_lens)}, max={max(asst_lens)}, avg={sum(asst_lens)/len(asst_lens):.0f}")


if csconda_converted:
    validate_dataset(csconda_converted, "CSConDa")
validate_dataset(ecommerce_converted, "Ecommerce")
validate_dataset(alpaca_converted, "Alpaca (filtered)")

In [ ]:
# Cell 13: Print 5 random samples from each for final human check
for name, data in [("CSConDa", csconda_converted), ("Ecommerce", ecommerce_converted), ("Alpaca", alpaca_converted)]:
    if not data:
        continue
    print(f"\n{'='*50}")
    print(f"  {name} - 5 random samples")
    print(f"{'='*50}")
    samples = random.sample(data, min(5, len(data)))
    for s in samples:
        for msg in s['messages']:
            if msg['role'] == 'system':
                continue
            prefix = 'USER' if msg['role'] == 'user' else 'ASST'
            print(f"  {prefix}: {msg['content'][:200]}")
        print()

---
## Step 5: Merge and split

In [ ]:
# Cell 14: Merge all datasets with source tracking
all_data = csconda_converted + ecommerce_converted + alpaca_converted
random.shuffle(all_data)

print(f"Total merged samples: {len(all_data)}")
print(f"\nSource distribution:")
source_counts = Counter(d['source'] for d in all_data)
for source, count in source_counts.most_common():
    print(f"  {source}: {count} ({count/len(all_data)*100:.1f}%)")

In [ ]:
# Cell 15: Train/Val/Test split (80/10/10)
n = len(all_data)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

train_data = all_data[:train_end]
val_data = all_data[train_end:val_end]
test_data = all_data[val_end:]

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

In [ ]:
# Cell 16: Save as JSONL
def save_jsonl(data: list[dict], path: Path) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"Saved {len(data)} samples to {path}")

save_jsonl(train_data, OUTPUT_DIR / "train.jsonl")
save_jsonl(val_data, OUTPUT_DIR / "val.jsonl")
save_jsonl(test_data, OUTPUT_DIR / "test.jsonl")

print(f"\nAll files saved to {OUTPUT_DIR}/")
print("Data preparation complete!")